# Phase 2V2 on Colab — seed-extension campaign (seeds 16 / 17 / 18)

**What this is:** the second sample of the Phase 2 experiment (`PHASE2_PLAN.md`). The nine
runs C1/C2/C3 × seeds 13/14/15 and the C0 baseline already exist and are consolidated; this
campaign trains **six new runs** — C1/C2/C3 × seeds **16/17/18** — on the **same frozen
seed-13 triples**, to upgrade every per-condition estimate from n=3 to n=6 and test that the
Phase 2 inversion (C1 ≈ 0.75 vs C2/C3 ≈ baseline) is seed-robust.

**What is frozen (identical to Phase 2, by construction):** corpus and splits, the three
triples files (= the manipulation, and the measured 42%/30% FN floors), the eval protocol
(same 415,416 shared test pairs, `max_f1_on_valid`, test scored once), the recipe (margin
0.10, batch 8, 32k cap, 1 epoch, max_len 512, fp16), and C0 (not trained — never re-run it).
A new training seed re-draws only batch order / dropout / fp16 nondeterminism; the model
still starts from the same pretrained GraphCodeBERT weights.

**Rules (from the Phase 2 parallel run):** disjoint per-VM slot assignments; interrupt only
between runs, after the push; trainers push artifacts, never the report; one consolidating VM
rebuilds the report and pushes to GitHub. C0 is skipped entirely, so this is **6 training
runs**, not 9 (≈ 4 VM-hours at 5 VMs).

Do **not** re-mine, do **not** bump `settings.VERSION`, do **not** point `EMBEDED_HF_SUBDIR`
anywhere new — the new run dirs join the existing tree of record.

## 0. Runtime and code checkout

In Colab choose **Runtime → Change runtime type → T4 GPU**. `REPO_REF` below is the same Arena work branch the Phase 2 runs used; the campaign cells refuse to run against a checkout whose `settings.VERSION` differs from the frozen artifacts.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e714-embeded'  # current Arena work branch (D1 gate rule, Phase 2 runner, Phase 3 runner); use main once merged
ROOT = '/content/EmbedEd'

def _git(*args, check=True):
    p = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if check and p.returncode:
        raise SystemExit(f"git {' '.join(args)} exited {p.returncode}\n{p.stdout}{p.stderr}")
    return p.stdout.strip()

if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT], check=True)
else:
    _git('fetch', 'origin', REPO_REF)
    ahead = _git('rev-list', '--count', 'FETCH_HEAD..HEAD', check=False)
    if ahead not in ('', '0'):
        raise SystemExit(
            f'{ahead} local commit(s) on this VM are not on origin/{REPO_REF}. Refusing to move the '
            'branch and orphan them: run notebook section 8 to push them (or `git rebase FETCH_HEAD` '
            'and push by hand), then re-run this cell.')
    _git('checkout', '-B', REPO_REF, 'FETCH_HEAD')
%cd /content/EmbedEd
print('Checked out:', _git('rev-parse', '--short', 'HEAD'))


## 1. Configure caches, install pinned dependencies, verify GPU, pull artifacts

Same environment as Phase 2 (identical pins). The pull is `--light`: trainers need the frozen triples and fragments, not the ~4.5 GB of existing checkpoints.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2 matplotlib==3.9.4
import torch, transformers, datasets, sentence_transformers, sklearn
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | transformers={transformers.__version__} | datasets={datasets.__version__} | sklearn={sklearn.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
# --light: trainers need fragments/triples/mining_summary, NOT the ~4.5 GB of checkpoints
hf_main(['pull', '--if-configured', '--light'])

## 2. Preflight: frozen artifacts and the offline test suite

The G1 gate is **not** re-run: it certified these exact triples once (hash-frozen in every run's `run_config.json`), and re-running it would append duplicate history to the report. Identity is instead re-proven after the campaign by appendix Part 2 (artifact hash-sets must still be one distinct set).

In [ ]:
from pathlib import Path
from embeded import settings as S
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1','C2','C3')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing Phase 1 artifacts; restore/regenerate and validate them before training:\n' + '\n'.join(missing))
print('Artifacts present for VERSION', S.VERSION)
import json as _json
_summary = S.ARTIFACTS / 'mining_summary.json'
if _summary.exists():
    _v = (_json.loads(_summary.read_text()).get('_run') or {}).get('version')
    print('mining_summary.json _run.version =', _v)
    if _v != S.VERSION:
        raise SystemExit(
            f'artifacts were mined by {_v!r} but this checkout is {S.VERSION!r}. '
            'embeded.train/evaluate refuse to run on them (exit 1, no traceback).\n'
            '  back up triples_C*.jsonl + mining_summary.json, then re-mine all three together:\n'
            '    python -m embeded.negatives --strategies random,bm25,semantic --force\n'
            '  and diff the new triples against the backup: if they are identical the blind\n'
            '  audit key still matches and its labels survive; if not, re-sample and re-label.')
else:
    print('mining_summary.json absent - run `python -m embeded.negatives` before training')
print('NOTE: confirm metadata/hashes and the shared anchors/positives/k; file existence alone is not an identity check.')

## 2.5 Audit labels: reuse the scored ones — no upload step

The blind audit was labelled and scored once in Phase 2 (100 pairs; FN floors 42% [29, 56]
for C2 and 30% [19, 44] for C3; second-pass recheck 100/100 on 2026-10-01). Both
`audit_labels.csv` and the key live in the HF artifact repo and arrive with the §1 pull, so
this notebook reads `artifacts/audit/audit_labels.csv` **directly** — there is no
`files.upload()` anywhere in this campaign. The cell validates the pulled labels, then
re-runs `scripts.audit_sample score`, which refuses unless the key still matches the triples
on disk — so a pass here is also proof that **this VM's triples are the exact ones the audit
scored**. Expect the recorded floors. Never run `make` in this campaign; if the files are
missing, the fix is a re-pull, not a re-label.

In [ ]:
# §2.5 — audit labels come straight from the artifact repo (scored in Phase 2)
import csv, subprocess, sys
from collections import Counter
from pathlib import Path

audit_dir = Path(ART) / 'audit'
labels_file, key_file = audit_dir / 'audit_labels.csv', audit_dir / 'audit_key.csv'
if not labels_file.exists() or not key_file.exists():
    raise SystemExit(
        f'{audit_dir}/ is incomplete (labels: {labels_file.exists()}, key: {key_file.exists()}) '
        '- the §1 pull was partial. Fix: re-run the pull '
        "(cell 4: hf_main(['pull', '--if-configured', '--light'])) and retry this cell. "
        'Do NOT hand-create or upload labels in this campaign - the scored file lives in the repo.')

rows = list(csv.DictReader(labels_file.open(encoding='utf-8-sig')))
ids = [r['id'] for r in rows]
labels = [r['label'].strip().lower() for r in rows]
assert ids == [f'P{i:03d}' for i in range(1, 101)], 'unexpected labels file (ids/order)'
assert set(labels) <= {'clone', 'not_clone', 'unsure'}, 'unexpected label vocabulary'
print('audit labels pulled:', dict(Counter(labels)))

# `score` refuses to run unless the key still matches the triples on disk, so this
# doubles as a triples-identity check for THIS VM (same gate as Phase 2's cell 12).
subprocess.run([sys.executable, '-m', 'scripts.audit_sample', 'score'], check=True)
print('audit reproduction OK - recorded floors: C2 42% [29, 56], C3 30% [19, 44] '
      '(pooled 36% [27, 46], blind 100-pair audit).')


## 3. Smoke test the runner on this VM

~3 minutes on a T4; writes to its own `runs/smoke_*` directories. The numbers mean nothing — it only proves train→checkpoint→evaluate works on this GPU before you spend the budget.

In [ ]:
import os, subprocess, sys
# Re-establish the environment: `-m embeded.*` resolves the package from the cwd and
# reads EMBEDED_ARTIFACTS from the environment, so a fresh runtime that skipped cell 1
# must not silently train against the in-repo default artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)   # os.chdir alone does not make `embeded` importable
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('Phase 2 runner modules are missing from this checkout - check REPO_REF in cell 0.')
# stderr is inherited, so the runner's own message (not just CalledProcessError) is on screen.
p = subprocess.run([sys.executable, '-m', 'embeded.train', '--condition', 'C1', '--seed', '13', '--smoke'])
if p.returncode:
    raise SystemExit(f'smoke run exited {p.returncode} - the reason is printed above. Do not start '
                     'the nine main runs until this passes.')
# trains a few steps, reloads the checkpoint, then evaluates on a tiny subset of
# both splits. Expect ~3 min on a T4; the numbers it prints mean nothing.

## 4. The campaign — train + evaluate this VM's slots (seeds 16/17/18)

**Before running:** edit `RUNS` in the cell below to this VM's slots from the assignment
table, so that across all VMs the slots are **disjoint** and together cover all nine. Each
run: train (~40 min on a T4) → evaluate → push the run dir to HF. The `[cache]` gate makes
re-running the cell safe (finished local runs skip); a mid-epoch interrupt simply means the
run restarts from the base model next time — interrupt **between** runs, after the push, as
in Phase 2. C0 is not in the list by design.

In [ ]:
import os, subprocess, sys
# Re-establish the environment (same preamble as Phase 2's runner cells): a fresh
# runtime that skipped the earlier cells must not train against the wrong artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

# ---- Phase 2V2 campaign: seeds 16/17/18 on the FROZEN seed-13 triples --------
# Nothing here re-mines or touches runs C*_13/14/15 or C0_13: new seeds land in
# NEW run directories (runs/C<cond>_<seed>), so this campaign is pure addition.
ALL_RUNS = [(c, s) for c in ('C1', 'C2', 'C3') for s in (16, 17, 18)]

# >>> THIS VM's slots. Edit before running, keep the sets DISJOINT across VMs. <<<
# Suggested 5-VM split (2/2/2/2/1):
#   VM1: [('C1',16), ('C1',17)]   VM2: [('C1',18), ('C2',16)]   VM3: [('C2',17), ('C2',18)]
#   VM4: [('C3',16), ('C3',17)]   VM5: [('C3',18)]
RUNS = ALL_RUNS            # <-- EDIT ME (default = this VM would do all nine)

print('this VM will run:', RUNS)
assert len(set(RUNS)) == len(RUNS), 'duplicate slots in this VM list'
assert set(RUNS) <= set(ALL_RUNS), 'slots outside the campaign'

# preflight: the frozen triples this VM needs must be present and version-correct
import json
from embeded import settings as S
summary = json.loads((S.ARTIFACTS / 'mining_summary.json').read_text())
if (summary.get('_run') or {}).get('version') != S.VERSION:
    raise SystemExit('stale artifacts: re-pull (cell 4) - NEVER re-mine for this campaign')
for c, s in RUNS:
    if not (S.ARTIFACTS / f'triples_{c}.jsonl').exists():
        raise SystemExit(f'missing triples_{c}.jsonl - re-pull (cell 4)')

def sh(*args):
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-m', *args]).returncode
    if rc:
        raise SystemExit(f'`python -m {args[0]}` exited {rc} - reason printed above.')

for condition, seed in RUNS:
    print(f'\n===== TRAIN {condition} seed={seed} =====', flush=True)
    sh('embeded.train', '--condition', condition, '--seed', str(seed))
    print(f'===== EVALUATE {condition} seed={seed} =====', flush=True)
    sh('embeded.evaluate', '--condition', condition, '--seed', str(seed))
    hf_main(['push', '--if-configured'])   # run dir only - NEVER --include-report from a trainer
print('\nCampaign slots done:', RUNS)
print('Re-running this cell is safe: finished runs print [cache] and skip.')


## 5. After this VM's slots finish

Push the run dirs once more (idempotent) and stop. Do not rebuild the report here — your pull sees only this VM's runs, and a partial table pushed now would be overwritten by the consolidator anyway.

In [ ]:
from scripts.hf_artifacts import main as hf_main
# Trainer VMs push ARTIFACTS ONLY. The report is rebuilt once, on the consolidating
# VM, from the union of all runs (standing rule from the Phase 2 parallel run).
hf_main(['push', '--if-configured'])
print('run dirs synced. Report/GitHub steps belong to the consolidating VM (next section).')


## 6. Consolidating VM only — rebuild the report from the union

Run this **after every trainer VM has finished and pushed**: pull `--light` (all 19 run dirs:
C0_13 + nine originals + nine extensions), rebuild the results table (it auto-aggregates all
discovered seeds, so the per-condition rows become n=6 trained + C0), render the
confusion-matrix board (auto-discovers every run dir; 19 panels), run the three
`sanitycheck.md` appendix blocks (they auto-discover seeds too), then push the report.

In [ ]:
import os, subprocess, sys
# Re-establish the environment (same preamble as Phase 2's runner cells): a fresh
# runtime that skipped the earlier cells must not train against the wrong artifacts dir.
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

# ---- CONSOLIDATING VM ONLY: rebuild the report from the union of all runs ----
# Run this after every trainer VM has finished and pushed. Pulling --light is
# enough for the table, the audit blocks and the figure (no checkpoints needed).
hf_main(['pull', '--if-configured', '--light'])

import subprocess, sys
p = subprocess.run([sys.executable, '-m', 'embeded.evaluate', '--results-table'])
if p.returncode:
    raise SystemExit('--results-table failed - reason printed above.')
print('table now aggregates every discovered run (original 13/14/15 + 16/17/18).')

# confusion board: auto-discovers every C0..C3 run dir with predictions.npz
import pathlib
from scripts.plot_confusion_matrices import main as cm_main
_runs = pathlib.Path(ART) / 'runs'
_out  = pathlib.Path(ROOT) / 'report' / 'figures' / 'phase2_confusion_matrices.png'
cm_main(['--runs-dir', str(_runs), '--out', str(_out)])

# sanitycheck appendix parts 1-3: run in a notebook cell by exec-ing the fenced
# blocks, or copy-paste them into fresh cells - they auto-discover seeds too.
hf_main(['push', '--if-configured', '--include-report'])
print('consolidation pushed. Now the GitHub step (final cell).')


## 7. Push the VM's commits back to GitHub

Same routine as Phase 2 §8: commit identity, rebase over remote, push with a PAT from the Colab Secrets panel (`GITHUB_TOKEN`).

In [ ]:
import json, os, subprocess, sys, urllib.error, urllib.request

ROOT = '/content/EmbedEd'
REF  = REPO_REF                 # cell 2
GIT_NAME  = ''                  # your GitHub username; '' = derive it from the PAT below
GIT_EMAIL = ''                  # the noreply address shown at https://github.com/settings/emails
COMMIT_DIRTY_REPORT = True      # commit tracked changes under report/ before rebasing
TOKEN_SECRET = 'GITHUB_TOKEN'   # Colab Secrets panel (key icon). No token -> stops before pushing.
API_USER = 'https://api.github.com/user'

def sh(*args, check=False):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(('GIT_AUTHOR_', 'GIT_COMMITTER_'))}   # these override user.email
    p = subprocess.run(['git', '-C', ROOT, *args], capture_output=True, text=True, env=env)
    shown = (p.stdout + p.stderr).strip()
    print(f"$ git {' '.join(args)} -> exit {p.returncode}")
    if shown:
        print('  ' + shown.replace('\n', '\n  '))
    if check and p.returncode:
        raise SystemExit(f'git {" ".join(args)} failed -- stopping, nothing was lost')
    return p.returncode, p.stdout          # raw: porcelain needs its leading status column

def read_token():                          # Colab Secrets first, then the environment
    try:
        from google.colab import userdata
        value = userdata.get(TOKEN_SECRET)
        if value:
            return value.strip()
    except Exception:
        pass
    return (os.environ.get(TOKEN_SECRET) or '').strip()

# 1. identity first: every commit and every rebase needs one, and a fresh VM has none
if not sh('config', '--get', 'user.email')[1].strip():
    name, email = GIT_NAME, GIT_EMAIL
    token = read_token()
    if not (name and email) and token:
        req = urllib.request.Request(API_USER, headers={'Authorization': f'Bearer {token}',
                                                        'Accept': 'application/vnd.github+json',
                                                        'User-Agent': 'embeded-colab'})
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                info = json.loads(resp.read().decode())
            if info.get('id') and info.get('login'):
                name = info['login']
                email = f"{info['id']}+{info['login']}@users.noreply.github.com"
                print(f"  derived the identity from the PAT: {name} <{email}>")
        except (urllib.error.URLError, urllib.error.HTTPError, OSError, ValueError) as exc:
            print(f'  could not reach {API_USER}: {exc.__class__.__name__}')
    if not (name and email):
        raise SystemExit(
            'No git identity on this VM and nothing to derive one from. Pick one:\n'
            f'  a) set GIT_NAME / GIT_EMAIL at the top of this cell, or\n'
            f'  b) add the PAT as a Colab Secret named {TOKEN_SECRET} (then it is derived above), or\n'
            f'  c) run:  !git -C {ROOT} config user.name "you"  and  '
            f'!git -C {ROOT} config user.email "ID+you@users.noreply.github.com"\n'
            'Nothing was changed; re-run this cell afterwards.')
    sh('config', 'user.name', name, check=True)
    sh('config', 'user.email', email, check=True)
    print('  commits will now be authored by: {} <{}>'.format(
        sh('config', '--get', 'user.name')[1].strip(),
        sh('config', '--get', 'user.email')[1].strip()))

# 2. tracked changes block a rebase; untracked strays usually do not
_, raw = sh('status', '--porcelain=v1')
entries = [(line[:2], line[3:].strip()) for line in raw.splitlines() if line.strip()]
tracked = [path for xy, path in entries if xy.strip() != '??']
untracked = [path for xy, path in entries if xy.strip() == '??']

# 3. commit tracked changes under report/ only; audit_key* is never staged
report = [p for p in tracked if p.startswith('report/') and 'audit_key' not in p]
other = [p for p in tracked if p not in report]
if other:
    raise SystemExit(f'tracked changes outside report/ {other} -- commit or stash them, then re-run')
if tracked:
    if not COMMIT_DIRTY_REPORT:
        raise SystemExit(f'tracked changes {tracked} but COMMIT_DIRTY_REPORT is False')
    sh('add', '--', *report, check=True)
    sh('commit', '-m', 'report: VM measurements + audit separability', check=True)

# 4. public repo: fetch needs no token
sh('fetch', 'origin', REF, check=True)

# 5. an untracked file only blocks the rebase if the remote adds that same path
for path in untracked:
    if sh('ls-tree', '--name-only', 'FETCH_HEAD', '--', path)[1].strip():
        raise SystemExit(f'{path} is untracked here but exists on the remote -- move or delete it, '
                         'then re-run')
    print(f'  (untracked, remote does not have it -> leaving it alone: {path})')

# 6. replay any local commit on top of the branch tip
if sh('rebase', 'FETCH_HEAD')[0]:
    raise SystemExit('rebase stopped -- your commits are safe. Either resolve the conflict and '
                     '`git rebase --continue`, or `git -C %s rebase --abort` to undo it.' % ROOT)
sh('log', '--oneline', '-4')

# 7. scripts.colab_git only exists after the rebase above
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.modules.pop('scripts.colab_git', None)  # a cached pre-rebase copy would shadow the new code
from scripts.colab_git import diagnose, read_secret, sync_and_push

print('\n--- diagnose ---')
for line in diagnose(ROOT, REF):
    print(line)

token = read_secret((TOKEN_SECRET,))
if not token:
    print(f'\nNo {TOKEN_SECRET} in Colab Secrets -> stopping before the push.')
    print('Add the fine-grained PAT as a Colab Secret named GITHUB_TOKEN, then re-run this cell:')
    print('the identity, the commit and the rebase are already done, so it goes straight to push.')
else:
    print('\n--- push ---')
    result = sync_and_push(ROOT, REF, token=token)
    for line in result['lines']:
        print(line)
    print(('ok  ' if result['ok'] else 'FAIL') + f" [{result['stage']}] {result['message']}")


## 8. Reading the results (what changes in the write-up)

- Per-condition rows become **mean ± sd over six seeds**; compare against the frozen n=3
  values (C1 0.753 ± 0.008; C2 0.254 ± 0.011; C3 0.273 ± 0.014). The finding survives if the
  gap stays ~3× with overlapping per-condition distributions.
- Appendix Part 1's expected readings apply per run; Part 2 must still print
  **one distinct artifact hash-set** (now across up to 10 VMs of history); Part 3 gains nine
  more rows — C2/C3 extensions should again show thresholds pinned at/near 1.0.
- `run_config.json` records `seed_list: [13, 14, 15]` from `settings.SEEDS` even for seed-16
  runs — cosmetic, ignore it; the `seed` field is authoritative. Do not bump `VERSION` to
  "fix" it.
- If any extension seed of C1 lands far below 0.70, or any C2/C3 seed far above 0.35: stop,
  check that run's appendix rows (steps=4000, no NaN, fingerprint OK) before interpreting.